# REINFORCE Demos

In [ ]:
import jax
import jax.numpy as jnp
import jax.random as jr
import flax.linen as nn
import optax
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation, patches
from IPython.display import HTML

%matplotlib inline
%config InlineBackend.figure_format='retina'

---
## 1. CartPole

Balance a pole on a cart by pushing left or right.

- **State** (4): cart position, velocity, pole angle, angular velocity
- **Actions**: push left or right
- **Reward**: +1 per step the pole stays up
- **Done**: pole past 12°, cart off track, or 500 steps

In [ ]:
# CartPole physics in pure JAX


def cartpole_step(state, action):
    x, x_dot, theta, theta_dot = state
    force = jnp.where(action == 1, 10.0, -10.0)
    cos_th, sin_th = jnp.cos(theta), jnp.sin(theta)
    total_mass, ml = 1.1, 0.05
    temp = (force + ml * theta_dot**2 * sin_th) / total_mass
    theta_acc = (9.8 * sin_th - cos_th * temp) / (0.5 * (4 / 3 - 0.1 * cos_th**2 / total_mass))
    x_acc = temp - ml * theta_acc * cos_th / total_mass
    new_state = jnp.array(
        [x + 0.02 * x_dot, x_dot + 0.02 * x_acc, theta + 0.02 * theta_dot, theta_dot + 0.02 * theta_acc]
    )
    done = (jnp.abs(new_state[0]) > 2.4) | (jnp.abs(new_state[2]) > 12 * jnp.pi / 180)
    return new_state, done


def record_cartpole(params, seed=0):
    key = jr.PRNGKey(seed)
    init_key, key = jr.split(key)
    state = jr.uniform(init_key, (4,), minval=-0.05, maxval=0.05)
    states = [np.array(state)]
    for _ in range(500):
        logits = policy.apply(params, state)
        key, subkey = jr.split(key)
        action = jr.categorical(subkey, logits)
        state, done = cartpole_step(state, action)
        states.append(np.array(state))
        if done:
            break
    return np.array(states)


def animate_cartpole(states, interval=20):
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.set_xlim(-2.5, 2.5)
    ax.set_ylim(-0.5, 1.5)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.plot([-2.5, 2.5], [0, 0], "k-", lw=1)
    cart = patches.FancyBboxPatch(
        (0, 0), 0.4, 0.2, boxstyle="round,pad=0.02", facecolor="#5c6bc0", edgecolor="#333", lw=1.5
    )
    ax.add_patch(cart)
    (pole,) = ax.plot([], [], color="#c62828", lw=3, solid_capstyle="round")
    txt = ax.text(0.02, 0.95, "", transform=ax.transAxes, fontsize=11, va="top")

    def update(i):
        x, _, th, _ = states[i]
        cart.set_x(x - 0.2)
        cart.set_y(-0.1)
        pole.set_data([x, x + np.sin(th)], [0.1, 0.1 + np.cos(th)])
        txt.set_text(f"step {i}")
        return cart, pole, txt

    anim = animation.FuncAnimation(fig, update, frames=len(states), interval=interval, blit=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())

In [ ]:
# Policy: state (4,) -> logits (2,)


class Policy(nn.Module):
    @nn.compact
    def __call__(self, x):
        x = nn.relu(nn.Dense(32)(x))
        x = nn.relu(nn.Dense(32)(x))
        return nn.Dense(2)(x)


policy = Policy()
params = policy.init(jr.PRNGKey(0), jnp.zeros(4))
print(f"P(left), P(right): {jax.nn.softmax(policy.apply(params, jnp.ones(4)))}")

In [ ]:
random_params = policy.init(jr.PRNGKey(4), jnp.zeros(4))
s = record_cartpole(random_params)
print(f"Random policy: {len(s)} steps")
animate_cartpole(s)

In [ ]:
# JIT'd trajectory collection


@jax.jit
def collect_trajectory(params, key):
    init_key, key = jr.split(key)
    state = jr.uniform(init_key, (4,), minval=-0.05, maxval=0.05)

    def body(carry, _):
        state, key, alive = carry
        logits = policy.apply(params, state)
        key, subkey = jr.split(key)
        action = jr.categorical(subkey, logits)
        new_state, done = cartpole_step(state, action)
        alive = alive & (~done)
        return (new_state, key, alive), (state, action, alive)

    _, (states, actions, alive) = jax.lax.scan(body, (state, key, jnp.bool_(True)), None, length=500)
    return states, actions, alive.sum() + 1


# REINFORCE loss
def reinforce_loss(params, states, actions, advantage, length):
    logits = jax.vmap(lambda s: policy.apply(params, s))(states)
    log_probs = jax.nn.log_softmax(logits)
    alp = log_probs[jnp.arange(len(actions)), actions]
    mask = jnp.arange(len(actions)) < length
    return -(alp * mask).sum() * advantage


grad_fn = jax.jit(jax.value_and_grad(reinforce_loss))

In [ ]:
def train_cartpole(n_episodes=2000, batch_size=20, lr=3e-3, seed=0):
    key = jr.PRNGKey(seed)
    params = policy.init(key, jnp.zeros(4))
    optimizer = optax.adam(lr)
    opt_state = optimizer.init(params)
    all_returns = []
    for ep in range(0, n_episodes, batch_size):
        keys = jr.split(key, batch_size + 1)
        key = keys[0]
        batch = [collect_trajectory(params, keys[i + 1]) for i in range(batch_size)]
        returns = jnp.array([l.astype(jnp.float32) for _, _, l in batch])
        baseline = returns.mean()
        all_returns.extend(returns.tolist())
        grads = jax.tree.map(jnp.zeros_like, params)
        for i, (states, actions, length) in enumerate(batch):
            _, g = grad_fn(params, states, actions, returns[i] - baseline, length)
            grads = jax.tree.map(lambda a, b: a + b, grads, g)
        grads = jax.tree.map(lambda g: g / batch_size, grads)
        updates, opt_state = optimizer.update(grads, opt_state)
        params = optax.apply_updates(params, updates)
        if (ep // batch_size) % 10 == 0:
            print(f"Episode {ep:4d} | mean return: {float(returns.mean()):.0f}")
    return params, all_returns


cartpole_params, cartpole_returns = train_cartpole()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(cartpole_returns, alpha=0.3, color="#5c6bc0", lw=0.8)
w = 20
sm = np.convolve(cartpole_returns, np.ones(w) / w, mode="valid")
ax.plot(range(w - 1, len(cartpole_returns)), sm, color="#5c6bc0", lw=2)
ax.axhline(500, color="#c62828", ls="--", lw=1, label="Max (500)")
ax.set_xlabel("Episode")
ax.set_ylabel("Return")
ax.set_title("REINFORCE on CartPole")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
random_params = policy.init(jr.PRNGKey(99), jnp.zeros(4))
s = record_cartpole(random_params)
print(f"Random policy: {len(s)} steps")
animate_cartpole(s)

In [ ]:
s = record_cartpole(cartpole_params)
print(f"Trained policy: {len(s)} steps")
animate_cartpole(s)

---
## 2. Charged particle navigation

A particle starts on the left and must reach the right side of a box filled with fixed charges.

- **State** (6): position $(x, y)$, velocity $(v_x, v_y)$, Coulomb force $(F_x, F_y)$
- **Actions**: thrust in 5 directions (none, right, up, left, down)
- **Reward**: rightward progress per step
- **Done**: reach right wall, leave the box, or 300 steps

In [ ]:
# Fixed charges + physics


def make_charges(n_charges=8, seed=42):
    rng = np.random.RandomState(seed)
    positions = rng.uniform(1.5, 8.5, size=(n_charges, 2))
    signs = rng.choice([-1.0, 1.0], size=n_charges)
    strengths = rng.uniform(3.0, 8.0, size=n_charges) * signs
    return jnp.array(positions), jnp.array(strengths)


CHARGE_POS, CHARGE_STR = make_charges()
THRUST_DIRS = jnp.array([[0, 0], [1, 0], [0, 1], [-1, 0], [0, -1.0]])


def coulomb_force(pos, charge_pos, charge_str):
    diff = pos - charge_pos
    dist_sq = jnp.sum(diff**2, axis=1) + 0.5
    return jnp.sum((charge_str / dist_sq)[:, None] * diff / jnp.sqrt(dist_sq)[:, None], axis=0)


def particle_step(state, action):
    pos, vel = state[:2], state[2:4]
    f_coulomb = coulomb_force(pos, CHARGE_POS, CHARGE_STR)
    vel = 0.98 * vel + 0.1 * (f_coulomb + THRUST_DIRS[action] * 2.0)
    new_pos = pos + 0.1 * vel
    reward = new_pos[0] - pos[0]
    done = (new_pos[0] > 10) | (new_pos[0] < 0) | (new_pos[1] < 0) | (new_pos[1] > 10)
    return jnp.concatenate([new_pos, vel, f_coulomb]), reward, done


def record_particle(params, seed=0):
    key = jr.PRNGKey(seed)
    init_key, key = jr.split(key)
    y0 = float(jr.uniform(init_key, (), minval=3.0, maxval=7.0))
    state = jnp.array([0.5, y0, 0.0, 0.0, 0.0, 0.0])
    positions = [np.array(state[:2])]
    for _ in range(300):
        logits = particle_policy.apply(params, state)
        key, subkey = jr.split(key)
        state, _, done = particle_step(state, int(jr.categorical(subkey, logits)))
        positions.append(np.array(state[:2]))
        if done:
            break
    return np.array(positions)


def animate_particle(positions, title=""):
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.set_aspect("equal")
    for k in range(len(CHARGE_STR)):
        c = "#c62828" if CHARGE_STR[k] > 0 else "#1565c0"
        ax.scatter(
            *CHARGE_POS[k], c=c, s=80 * abs(CHARGE_STR[k]) / 8, zorder=5, edgecolors="white", linewidth=1, alpha=0.7
        )
    (trail,) = ax.plot([], [], color="#5c6bc0", lw=1.5, alpha=0.5)
    dot = ax.scatter([], [], c="#5c6bc0", s=80, zorder=10, edgecolors="white", linewidth=1.5)
    txt = ax.text(0.02, 0.97, "", transform=ax.transAxes, fontsize=11, va="top")
    ax.set_title(title)
    ax.axvline(10, color="gold", ls="--", lw=1.5, alpha=0.5)

    def update(i):
        trail.set_data(positions[: i + 1, 0], positions[: i + 1, 1])
        dot.set_offsets([positions[i]])
        txt.set_text(f"step {i}  x={positions[i, 0]:.1f}")
        return trail, dot, txt

    anim = animation.FuncAnimation(fig, update, frames=len(positions), interval=30, blit=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())

In [ ]:
# Visualize the force field
fig, ax = plt.subplots(figsize=(6, 6))
xs, ys = np.linspace(0, 10, 40), np.linspace(0, 10, 40)
Fx, Fy = np.zeros((40, 40)), np.zeros((40, 40))
for i, x in enumerate(xs):
    for j, y in enumerate(ys):
        f = coulomb_force(jnp.array([x, y]), CHARGE_POS, CHARGE_STR)
        Fx[j, i], Fy[j, i] = f[0], f[1]
ax.streamplot(xs, ys, Fx, Fy, color=np.sqrt(Fx**2 + Fy**2), cmap="coolwarm", density=1.5, linewidth=0.8, arrowsize=0.8)
for k in range(len(CHARGE_STR)):
    c = "#c62828" if CHARGE_STR[k] > 0 else "#1565c0"
    ax.scatter(*CHARGE_POS[k], c=c, s=80 * abs(CHARGE_STR[k]) / 8, zorder=5, edgecolors="white", linewidth=1)
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_aspect("equal")
ax.set_title("Coulomb force field")
ax.axvline(0.5, color="green", ls="--", lw=1.5, label="Start")
ax.axvline(10, color="gold", ls="--", lw=1.5, label="Goal")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# Particle policy + JIT'd training


class ParticlePolicy(nn.Module):
    @nn.compact
    def __call__(self, x):
        x = nn.relu(nn.Dense(64)(x))
        x = nn.relu(nn.Dense(64)(x))
        return nn.Dense(5)(x)


particle_policy = ParticlePolicy()


@jax.jit
def collect_particle_traj(params, key):
    init_key, key = jr.split(key)
    y0 = jr.uniform(init_key, (), minval=2.0, maxval=8.0)
    state = jnp.array([0.5, y0, 0.0, 0.0, 0.0, 0.0])

    def body(carry, _):
        state, key, alive, total_r = carry
        logits = particle_policy.apply(params, state)
        key, subkey = jr.split(key)
        action = jr.categorical(subkey, logits)
        new_state, reward, done = particle_step(state, action)
        alive = alive & (~done)
        total_r = total_r + reward * alive
        return (new_state, key, alive, total_r), (state, action, alive)

    (_, _, _, total_r), (states, actions, alive) = jax.lax.scan(
        body, (state, key, jnp.bool_(True), 0.0), None, length=300
    )
    return states, actions, total_r, alive.sum() + 1


def particle_loss(params, states, actions, advantage, length):
    logits = jax.vmap(lambda s: particle_policy.apply(params, s))(states)
    alp = jax.nn.log_softmax(logits)[jnp.arange(len(actions)), actions]
    return -(alp * (jnp.arange(len(actions)) < length)).sum() * advantage


particle_grad_fn = jax.jit(jax.value_and_grad(particle_loss))

In [ ]:
random_particle_params = particle_policy.init(jr.PRNGKey(99), jnp.zeros(6))
pos = record_particle(random_particle_params, seed=3)
print(f"Random: {len(pos)} steps, x={pos[-1, 0]:.1f}")
animate_particle(pos, "Random policy")

In [ ]:
def train_particle(n_episodes=4000, batch_size=30, lr=3e-3, seed=0):
    key = jr.PRNGKey(seed)
    params = particle_policy.init(key, jnp.zeros(6))
    optimizer = optax.adam(lr)
    opt_state = optimizer.init(params)
    all_returns = []
    for ep in range(0, n_episodes, batch_size):
        keys = jr.split(key, batch_size + 1)
        key = keys[0]
        batch = [collect_particle_traj(params, keys[i + 1]) for i in range(batch_size)]
        returns = jnp.array([r for _, _, r, _ in batch])
        baseline = returns.mean()
        all_returns.extend(returns.tolist())
        grads = jax.tree.map(jnp.zeros_like, params)
        for i, (states, actions, R, length) in enumerate(batch):
            _, g = particle_grad_fn(params, states, actions, returns[i] - baseline, length)
            grads = jax.tree.map(lambda a, b: a + b, grads, g)
        grads = jax.tree.map(lambda g: g / batch_size, grads)
        updates, opt_state = optimizer.update(grads, opt_state)
        params = optax.apply_updates(params, updates)
        if (ep // batch_size) % 10 == 0:
            print(f"Episode {ep:4d} | mean return: {float(returns.mean()):.2f}")
    return params, all_returns


particle_params, particle_returns = train_particle()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(particle_returns, alpha=0.2, color="#5c6bc0", lw=0.5)
w = 30
sm = np.convolve(particle_returns, np.ones(w) / w, mode="valid")
ax.plot(range(w - 1, len(particle_returns)), sm, color="#5c6bc0", lw=2)
ax.axhline(0, color="gray", ls=":", lw=0.8)
ax.set_xlabel("Episode")
ax.set_ylabel("Return (rightward distance)")
ax.set_title("REINFORCE: Charged Particle Navigation")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
random_particle_params = particle_policy.init(jr.PRNGKey(99), jnp.zeros(6))
pos = record_particle(random_particle_params, seed=3)
print(f"Random: {len(pos)} steps, x={pos[-1, 0]:.1f}")
animate_particle(pos, "Random policy")

In [ ]:
pos = record_particle(particle_params, seed=7)
print(f"Trained: {len(pos)} steps, x={pos[-1, 0]:.1f}")
animate_particle(pos, "Trained policy")

In [ ]:
# Side-by-side: multiple trajectories
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, p, title in [(axes[0], random_particle_params, "Random"), (axes[1], particle_params, "Trained")]:
    for k in range(len(CHARGE_STR)):
        c = "#c62828" if CHARGE_STR[k] > 0 else "#1565c0"
        ax.scatter(
            *CHARGE_POS[k], c=c, s=60 * abs(CHARGE_STR[k]) / 8, zorder=5, edgecolors="white", linewidth=0.8, alpha=0.6
        )
    for seed in range(8):
        pos = record_particle(p, seed=seed)
        ax.plot(pos[:, 0], pos[:, 1], lw=1, alpha=0.6)
        ax.scatter(pos[-1, 0], pos[-1, 1], s=20, zorder=6, color="black")
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.set_aspect("equal")
    ax.set_title(title)
    ax.axvline(10, color="gold", ls="--", lw=1, alpha=0.5)
plt.tight_layout()
plt.show()